In [1]:
from datasets import load_dataset
from transformers import BertForSequenceClassification, BertTokenizer
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import classification_report, confusion_matrix
import torch
import pandas as pd

In [2]:
dataset = load_dataset("amazon_polarity")
test_data = dataset["test"].shuffle(seed=42).select(range(500))

In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'content'],
        num_rows: 3600000
    })
    test: Dataset({
        features: ['label', 'title', 'content'],
        num_rows: 400000
    })
})

In [4]:
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):  
        self.encoding = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
     )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self,idx):
        return{
           "input_ids":   self.encoding["input_ids"][idx],
           "attention_mask":  self.encoding["attention_mask"][idx],
           "label":            self.labels[idx]
       }

In [5]:
model = BertForSequenceClassification.from_pretrained("./bert-sentiment-final")
tokenizer = BertTokenizer.from_pretrained("./bert-sentiment-final")
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [6]:
texts = [str(x) for x in test_data["content"]]
labels = list(test_data["label"])

review_dataset = ReviewDataset(texts, labels, tokenizer)
loader = DataLoader(review_dataset, batch_size=32)

In [7]:
all_preds  = []
all_labels = []

print("Running predictions...")
with torch.no_grad():
    for i, batch in enumerate(loader):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        output = model(input_ids=input_ids, attention_mask=attention_mask)
        preds  = torch.argmax(output.logits, dim=1).cpu().tolist()

        all_preds.extend(preds)
        all_labels.extend(batch["label"].tolist())

        if (i + 1) % 5 == 0:
            print(f"  Processed {(i+1)*32} reviews...")


Running predictions...
  Processed 160 reviews...
  Processed 320 reviews...
  Processed 480 reviews...


In [8]:
print("\n Classification Report  ")
print(classification_report(
    all_preds, all_labels,
    target_names=["Neagtive","Positive"]
))

print("Confusion Matrix")
cm = confusion_matrix(all_preds, all_labels)
cm_df = pd.DataFrame(
    cm,
    index = ["Actual Neagtive", "Actual Positive"],
    columns = ["Predicted Negative", "Predicted Positive"]
)
print(cm_df)


 Classification Report  
              precision    recall  f1-score   support

    Neagtive       0.95      0.85      0.90       282
    Positive       0.83      0.94      0.88       218

    accuracy                           0.89       500
   macro avg       0.89      0.90      0.89       500
weighted avg       0.90      0.89      0.89       500

Confusion Matrix
                 Predicted Negative  Predicted Positive
Actual Neagtive                 240                  42
Actual Positive                  13                 205


In [9]:
print("\n── Examples where model was wrong ────────────────")
wrong = [(texts[i], all_labels[i], all_preds[i])
         for i in range(len(all_labels)) if all_labels[i] != all_preds[i]]

label_map = {0: "Negative", 1: "Positive"}
for text, true, pred in wrong[:3]:
    print(f"  Text    : {text[:80]}...")
    print(f"  True    : {label_map[true]}")
    print(f"  Predicted: {label_map[pred]}")
    print()


── Examples where model was wrong ────────────────
  Text    : The product works fine. I ordered the more exprensive one after I read reviews f...
  True    : Positive
  Predicted: Negative

  Text    : 1.Music-I don't like it,but too overplayed for this CD-2/5 stars2.Most Girls-I l...
  True    : Positive
  Predicted: Negative

  Text    : The other plastic garages and service stations that we have purchased in the pas...
  True    : Positive
  Predicted: Negative

